[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhangdongming0607/ai-memory-course/blob/main/07-hands-on-mem0.ipynb)

> 点击上方按钮，在 Google Colab 中打开本课，可以直接运行代码，无需本地环境。

# 第 7 课：实战 — 用 Mem0 搭建联系人记忆

> 学完这节课，你会用 Mem0 库亲手搭建一个联系人记忆系统，理解记忆的存储、检索和自动更新。

---

## 7.1 Mem0 快速入门

[Mem0](https://github.com/mem0ai/mem0) 是一个开源的 AI 记忆层库。它帮你做了这些事：

1. 从对话中**自动提取**关键信息
2. 把信息**向量化**存储（还记得前面课讲的 embedding 吗？）
3. 查询时**语义搜索**相关记忆
4. 信息冲突时**自动更新**

核心 API 只有 3 个：`add`、`search`、`get_all`。

先安装：

In [ ]:
!pip install mem0ai -q

创建 Memory 实例。Mem0 本地模式依赖 OpenAI SDK 来调用 LLM 和生成 embedding，所以需要设置 `OPENAI_API_KEY` 环境变量：

> **注意**：如果你使用其他 LLM 提供商（如 Azure OpenAI、Ollama 等），可以通过配置 `llm` 和 `embedder` 参数来指定，详见 [Mem0 文档](https://docs.mem0.ai/open-source/llms)。这里我们用最简单的默认配置。

In [ ]:
import os
from mem0 import Memory

# 确保设置了 OpenAI API Key
# os.environ["OPENAI_API_KEY"] = "sk-xxx"  # 取消注释并填入你的 key

# 创建 Memory 实例，使用默认配置（本地模式）
# 默认使用 OpenAI 的 gpt-4o-mini 作为 LLM，text-embedding-3-small 作为 embedding 模型
memory = Memory()
print("Memory 实例创建成功！")

试试最基本的 `add` 和 `search`：

In [ ]:
# 添加一条记忆，user_id 用来区分不同用户
result = memory.add("小明喜欢打羽毛球，每周三晚上都会去", user_id="test_user")
print("添加结果：")
print(result)

In [ ]:
# 搜索记忆
results = memory.search("小明有什么爱好？", user_id="test_user")
print("搜索结果：")
for r in results.get("results", []):
    print(f"  - {r['memory']}（相似度: {r.get('score', 'N/A')}）")

In [ ]:
# 查看所有记忆
all_memories = memory.get_all(user_id="test_user")
print("所有记忆：")
for m in all_memories.get("results", []):
    print(f"  - {m['memory']}")

就这么简单。3 行代码就完成了「存储 - 检索」的完整流程。

---

## 7.2 存储联系人对话

现在模拟一个真实场景：用户上传了和小明的聊天记录，我们用 Mem0 来提取和存储记忆。

In [ ]:
# 先重置一下，清除之前测试的数据
memory.reset()
memory = Memory()

# 模拟一段和小明的聊天记录
chat_messages = [
    {"role": "user", "content": "小明，周末去打羽毛球吗？"},
    {"role": "assistant", "content": "好呀！周六下午怎么样？我最近换了新球拍，正想试试"},
    {"role": "user", "content": "行，老地方见。对了，你上次说要学吉他，报班了吗？"},
    {"role": "assistant", "content": "报了报了，周二和周四晚上的课。老师说我有天赋哈哈"},
    {"role": "user", "content": "厉害了！下次聚会给我们表演一个"},
    {"role": "assistant", "content": "等我学完《小星星》先吧。话说下周四是我生日，你来吗？想去那家新开的日料店"},
    {"role": "user", "content": "必须来！日料好评，我请你"},
]

# 这里 user_id 代表当前用户，metadata 可以记录关于对话的额外信息
result = memory.add(
    chat_messages,
    user_id="my_user",
    metadata={"contact": "小明"},
)

print("Mem0 自动提取的记忆：")
for item in result.get("results", []):
    print(f"  * {item.get('memory', item)}")

注意看 Mem0 做了什么：

- 它没有原封不动地存聊天记录
- 而是**自动提取**了关键信息，比如「小明喜欢打羽毛球」「小明在学吉他」「小明生日在下周四」
- 这和你们系统里「记忆提取」那一步做的事情是一样的！

现在试试搜索：

In [ ]:
# 搜索和小明爱好相关的记忆
results = memory.search("小明的兴趣爱好是什么？", user_id="my_user")
print("搜索「小明的兴趣爱好」：")
for r in results.get("results", []):
    print(f"  - {r['memory']}")

print()

# 搜索和小明生日相关的记忆
results = memory.search("小明的生日是什么时候？", user_id="my_user")
print("搜索「小明的生日」：")
for r in results.get("results", []):
    print(f"  - {r['memory']}")

语义搜索的威力：你搜「兴趣爱好」，它能匹配到「打羽毛球」和「学吉他」——因为它理解这些都是爱好。

---

## 7.3 记忆的自动更新

真实场景中，人的情况会变化。Mem0 怎么处理信息冲突？

In [ ]:
# 先清空，从头演示
memory.reset()
memory = Memory()

# 第一次：小明喜欢喝咖啡
memory.add("小明每天早上都要喝一杯美式咖啡，是个重度咖啡爱好者", user_id="my_user")

print("=== 第一次添加后 ===")
all_mem = memory.get_all(user_id="my_user")
for m in all_mem.get("results", []):
    print(f"  - {m['memory']}")

In [ ]:
# 第二次：小明改喝茶了
result = memory.add("小明最近戒了咖啡，改喝绿茶了，说是为了健康", user_id="my_user")

print("=== 更新操作详情 ===")
print(result)

print("\n=== 第二次添加后，当前所有记忆 ===")
all_mem = memory.get_all(user_id="my_user")
for m in all_mem.get("results", []):
    print(f"  - {m['memory']}")

观察结果：

- Mem0 **没有**简单地新增一条「小明喝茶」的记忆
- 而是识别出这和之前的「小明喝咖啡」是**冲突**的
- 自动把旧记忆**更新**成了新版本

这就是 Mem0 的核心能力之一：**自动冲突检测与合并**。

再来看一个补充信息（不冲突）的例子：

In [ ]:
# 补充新信息：不和之前的冲突
memory.add("小明养了一只橘猫叫橘子，已经养了两年了", user_id="my_user")

print("=== 补充信息后，当前所有记忆 ===")
all_mem = memory.get_all(user_id="my_user")
for m in all_mem.get("results", []):
    print(f"  - {m['memory']}")

不冲突的信息会被正常新增，冲突的信息会被更新。这个行为和你们系统的「差分合并」理念类似，但实现方式不同——Mem0 完全依赖 LLM 来判断冲突，你们系统用了更精细的规则。

---

## 7.4 搭建一个简化版联系人记忆系统

现在把前面学的串起来，搭建一个完整的 mini demo，模拟你们系统的核心流程：

1. 输入聊天记录 → 记忆提取并存储
2. 用户要回复联系人 → 检索相关记忆
3. 将记忆注入 prompt → 生成有「人情味」的回复

In [ ]:
from openai import OpenAI
from mem0 import Memory

# --- 初始化 ---
client = OpenAI()
memory = Memory()

def ingest_chat(contact_name: str, messages: list):
    """步骤 1：导入聊天记录，Mem0 自动提取记忆"""
    result = memory.add(messages, user_id="me", metadata={"contact": contact_name})
    extracted = [item.get("memory", str(item)) for item in result.get("results", [])]
    print(f"从和 {contact_name} 的对话中提取了 {len(extracted)} 条记忆")
    return extracted

def generate_reply(contact_name: str, recent_message: str) -> str:
    """步骤 2+3：检索记忆 -> 注入 prompt -> 生成回复"""
    # 步骤 2：检索与当前话题相关的记忆
    memories = memory.search(recent_message, user_id="me")
    memory_text = "\n".join(
        f"- {m['memory']}" for m in memories.get("results", [])
    )

    # 步骤 3：把记忆注入 System Prompt，让 AI 生成贴心的回复
    system_prompt = f"""你是一个帮用户生成聊天回复的助手。
根据以下关于联系人 {contact_name} 的记忆，生成自然、得体的回复。

【关于 {contact_name} 的记忆】
{memory_text if memory_text else '暂无记忆'}

要求：语气自然随意，像朋友间聊天。只输出回复内容，不要解释。"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": f"{contact_name} 说：{recent_message}\n\n请帮我生成回复："},
        ],
        temperature=0.7,
    )
    return response.choices[0].message.content

print("联系人记忆系统初始化完成！")

In [ ]:
# === 实际使用 ===

# 步骤 1：导入和小明的聊天记录
ingest_chat("小明", [
    {"role": "user", "content": "小明，周末去打羽毛球吗？"},
    {"role": "assistant", "content": "好呀！我最近换了把尤尼克斯的新拍，手感特别好"},
    {"role": "user", "content": "哈哈不错，对了你吉他学得怎么样？"},
    {"role": "assistant", "content": "能弹两首了！周四有个小型演出，你来捧场呗"},
    {"role": "user", "content": "必须去！到时候给你录视频"},
    {"role": "assistant", "content": "好哥们！对了下个月 15 号我生日，约一波？想去那家新开的日料"},
])

print("\n--- 当前所有记忆 ---")
all_mem = memory.get_all(user_id="me")
for m in all_mem.get("results", []):
    print(f"  - {m['memory']}")

In [ ]:
# 步骤 2+3：小明发来新消息，生成带有记忆的回复
new_message = "周末有空吗？好久没聚了"

print(f"小明说：{new_message}")
print()

reply = generate_reply("小明", new_message)
print(f"建议回复：{reply}")

In [ ]:
# 再试一条 —— 这次和生日相关
new_message_2 = "生日那天你能早点到吗？我订了六点的位子"

print(f"小明说：{new_message_2}")
print()

reply_2 = generate_reply("小明", new_message_2)
print(f"建议回复：{reply_2}")

对比一下有记忆和没记忆的区别：

- **没有记忆**：AI 只能生成通用回复，比如「好的没问题」
- **有记忆**：AI 知道小明喜欢日料、在学吉他，回复会更贴心，比如「六点没问题！生日快乐呀，日料店是上次说的那家吗？」

这就是记忆系统的价值——让 AI 的回复有**温度**。

---

## 7.5 和你们系统的对比

我们用 Mem0 大约 50 行代码就实现了一个能用的联系人记忆系统。回顾一下核心代码只用了 3 个 API：

```python
memory.add(messages, user_id=...)      # 存储记忆
memory.search(query, user_id=...)      # 检索记忆
memory.get_all(user_id=...)            # 查看所有记忆
```

而你们系统用了上百行代码来做类似的事情。为什么？

### Mem0 做不到的事情

| 能力 | Mem0 | 你们的系统 |
|------|------|----------|
| 基础记忆存取 | 3 行搞定 | 需要更多代码 |
| 按层级分类存储 | 所有记忆混在一起 | 分 fact / summary / tips 多层存储，结构更清晰 |
| 手动编辑保护 | 用户手动改了记忆，下次 add 可能覆盖 | 差分合并策略，保护手动编辑的内容 |
| 记忆冷却策略 | 无（每次 add 都触发 LLM） | 有冷却时间，避免频繁调用 LLM |
| 成本控制 | 每次操作都调 LLM | 批量处理、缓存策略，控制 API 调用量 |
| 记忆可解释性 | 黑盒（LLM 决定提取什么） | 可以精细控制提取规则 |

### 什么时候该用哪个？

In [ ]:
# 用表格直观展示
comparison = """
+-----------------------------------+--------------------+--------------------+
|           场景                    |  Mem0 等简单方案    |  定制方案           |
+-----------------------------------+--------------------+--------------------+
| 个人项目 / 原型验证               |  推荐              |  杀鸡用牛刀         |
| 用户量 < 1000                    |  够用              |  可以但没必要       |
| 用户量 > 10000                   |  成本和性能有风险    |  推荐              |
| 需要精细控制记忆格式              |  做不到             |  推荐              |
| 需要保护用户手动编辑              |  做不到             |  推荐              |
| 需要多层记忆结构                  |  需要自己封装       |  原生支持           |
| 快速上线 MVP                     |  半天搞定           |  至少一周           |
+-----------------------------------+--------------------+--------------------+
"""
print(comparison)

### 一个实际的决策思路

1. **先用 Mem0 跑通 MVP**：验证「记忆」这个功能对用户是否有价值
2. **遇到瓶颈再定制**：当出现以下信号时考虑自建：
   - 用户反馈「AI 记错了」或「AI 忘了我改的内容」
   - API 成本增长过快
   - 需要更复杂的记忆分类
3. **渐进式替换**：不需要一次性重写，可以先替换记忆存储层，再替换提取策略

你们的系统走到了第 2 步——因为有明确的产品需求（多层记忆、手动编辑保护），所以选择了定制方案。这是合理的技术演进路径。

---

## 7.6 本课小结

| 内容 | 学到了什么 |
|------|----------|
| Mem0 基础 | `add` / `search` / `get_all` 三个 API 搞定记忆存取 |
| 自动提取 | Mem0 能从对话中自动提取关键信息，不需要手写规则 |
| 冲突更新 | 信息矛盾时，Mem0 会自动更新而不是重复存储 |
| 完整流程 | 聊天记录 → 记忆提取 → 检索注入 → 生成回复，50 行代码搞定 |
| 方案选型 | 简单场景用 Mem0，复杂需求（分层、保护编辑、成本控制）用定制方案 |

### 关键收获

你们系统做的事情并不神秘——Mem0 用 3 个 API 就能实现 80% 的功能。但剩下 20% 的精细化（分层存储、手动编辑保护、冷却策略）才是产品竞争力所在。理解了简单方案，你才能真正理解定制方案**为什么要那样设计**。

### 下节预告

下一课我们会深入你们系统的实际代码，看看那些「Mem0 做不到」的部分到底是怎么实现的。